# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/jayanthGowda1718/ml-internship-work/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

One row = one content page (content_id), from an anonymized 90-day trailing window (impressions_90d, clicks_90d, etc.), with an additional last-30d vs. prev-30d split used for trend comparison. This is a single snapshot cut, not a live/streaming feed.

In [11]:
import pandas as pd
url = "https://raw.githubusercontent.com/jayanthGowda1718/ml-internship-work/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(url)

In [12]:
print(df.shape)
print(df["content_id"].nunique())


(30000, 44)
30000


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Feature (used to predict): search_volume, competition, competition_level, cpc, content_type, main_intent, word_count, char_count, impressions_90d, sessions_90d, users_90d, content_age_days, avg_position, freshness_tier, word_count_tier

Label (target): ctr, engagement_rate (used to build the proxy target needs_engagement_fix)

Context (identifiers, not predictive): content_id, client_id

Excluded:
- clicks_90d, engaged_sessions_90d, scroll_events_90d, scroll_rate — excluded because these are downstream of the label (ctr and engagement_rate are derived directly from clicks and engagement counts), so including them would leak the answer into the features.
- position_tier, impression_tier, trend_direction, trend_pct — excluded because these are pre-computed summary/tier fields likely derived from the same metrics we're trying to predict, risking leakage.
- ai_sessions_90d, ai_traffic_pct — excluded for this lane since they belong to the ai_opportunity lane, not engagement_fix.
- provider_used, model_used — excluded as internal tooling metadata, not a real signal about page performance.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

Each claim from the fields/contract above is checked with a query below: the grain check confirms one row = one content_id, the missing-values check confirms which fields are reliable enough to use, the describe() checks confirm feature ranges are sane, and the window check confirms how the 30d/90d periods relate to each other.

In [14]:
# Grain check
print(df.shape)
print(df["content_id"].nunique())


(30000, 44)
30000


In [15]:
# Missing values check
df.isnull().sum().sort_values(ascending=False).head(15)

,0
provider_used,21438
word_count,7699
char_count,7699
word_count_tier,7699
char_count_tier,7699
model_used,5733
trend_pct,3388
competition_level,2610
search_volume,2468
cpc,2468


In [16]:
# Feature columns actually exist and have sane ranges
df[["search_volume", "cpc", "word_count", "avg_position", "impressions_90d"]].describe()

,search_volume,cpc,word_count,avg_position,impressions_90d
count,27532.000000,27532.000000,22301.000000,30000.00000,30000.000000
mean,158.882391,0.485342,3107.760325,16.34238,5200.366300
std,1518.270825,2.101560,1452.382598,15.21679,16838.019547
min,0.000000,0.000000,8.000000,0.00000,1.000000
25%,0.000000,0.000000,2413.000000,6.20000,81.000000
50%,10.000000,0.000000,2877.000000,10.80000,731.000000
75%,20.000000,0.000000,3666.000000,22.30000,3615.250000
max,74000.000000,100.360000,9546.000000,245.00000,517715.000000


In [17]:
# Confirm label columns aren't duplicated as features (leakage check)
df[["ctr", "engagement_rate", "clicks_90d", "engaged_sessions_90d"]].describe()

,ctr,engagement_rate,clicks_90d,engaged_sessions_90d
count,30000.000000,30000.000000,30000.000000,30000.000000
mean,0.510733,2.534520,16.097333,0.991933
std,3.279162,8.310096,75.076958,4.359576
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000
50%,0.070000,0.000000,1.000000,0.000000
75%,0.290000,1.350000,7.000000,1.000000
max,100.000000,100.000000,4178.000000,290.000000


In [18]:
# Confirm the 30d vs 90d windows overlap (last_30d is a subset of the 90d window, not additive)
print(df["impressions_last_30d"].sum(), df["impressions_90d"].sum())

42871762 156010989


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

This data can never tell you: causal impact of any specific fix — it's observational, not an A/B test, so we can't say a change *caused* a CTR improvement, only that certain patterns correlate with it. It's a single 90-day snapshot, so it can't reveal longer seasonality or multi-quarter trends. Early-period pages likely have unbalanced history — fewer days_with_impressions and days_with_sessions than older pages — meaning their metrics are noisier and less trustworthy. The impressions_last_30d / impressions_prev_30d windows are subsets of the 90d window, not independent periods, so they can't be summed or compared as if they were separate. The data is anonymized (no query text, titles, or URLs), so it can explain *what* is underperforming numerically but never *why* in content terms — that judgment still needs a human reviewing the actual page.

In [19]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.